# Ko-Binoculars: 한국어 LLM 생성 텍스트 제로샷 탐지

Binoculars(Hans et al., ICML 2024)의 모델 쌍을 한국어 Polyglot-Ko로 교체하여
KatFish 데이터셋에서 한국어 LLM 생성 텍스트를 탐지합니다.

**실험 목록:**
1. 환경 설정 및 데이터 로드
2. 모델 로드 (영어 GPT2 쌍, 한국어 Polyglot 쌍)
3. 실험 1: 영어 모델 쌍 (베이스라인)
4. 실험 2: 한국어 모델 쌍 Ko-Binoculars (전체 데이터)
5. 실험 3: 장르별 분석
6. 실험 4: Ablation Study (모델 쌍 크기 조합)
7. 최종 결과 정리

## 셀 1: 설치

In [1]:
!pip install transformers datasets scikit-learn accelerate -q

## 셀 2: 임포트 및 GPU 확인

In [2]:
import torch
import numpy as np
import pandas as pd
import json
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')
print(f'torch 버전: {torch.__version__}')

디바이스: cuda
torch 버전: 2.10.0+cu128


## 셀 3: KatFish 데이터셋 로드

In [3]:
# KatFish GitHub 클론
!git clone https://github.com/Shinwoo-Park/detecting_llm_generated_korean_text_through_linguistic_analysis.git katfishnet
print(os.listdir('katfishnet/katfish_dataset'))

Cloning into 'katfishnet'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 41 (delta 21), reused 30 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 2.95 MiB | 22.89 MiB/s, done.
Resolving deltas: 100% (21/21), done.
['abstract.jsonl', 'essay.jsonl', 'poetry.jsonl']


In [4]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

essay    = load_jsonl('katfishnet/katfish_dataset/essay.jsonl')
abstract = load_jsonl('katfishnet/katfish_dataset/abstract.jsonl')
poetry   = load_jsonl('katfishnet/katfish_dataset/poetry.jsonl')

# 전체 데이터 합치기
all_data = essay + abstract + poetry
df = pd.DataFrame(all_data)

print(f'essay: {len(essay)}개')
print(f'abstract: {len(abstract)}개')
print(f'poetry: {len(poetry)}개')
print(f'전체: {len(df)}개')
print(f'\nlabel 분포:')
print(df['label'].value_counts())

essay: 771개
abstract: 378개
poetry: 945개
전체: 2094개

label 분포:
label
1    1624
0     470
Name: count, dtype: int64


## 셀 4: PPL / Binoculars 함수 정의

In [5]:
def compute_ppl(text, model, tokenizer, device, max_length=256):
    """단순 PPL 계산"""
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=max_length).to(device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs['input_ids']).loss
    return torch.exp(loss).item()


def compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device, max_length=256):
    """
    Binoculars Score = log PPL(observer) / log CrossPPL(observer, performer)
    낮을수록 AI 생성 텍스트
    """
    # Observer PPL (log 공간)
    inp_o = tok_obs(text, return_tensors='pt', truncation=True,
                    max_length=max_length).to(device)
    with torch.no_grad():
        loss_o = m_obs(**inp_o, labels=inp_o['input_ids']).loss.item()

    # Performer logits → Observer로 cross entropy 계산
    inp_p  = tok_per(text, return_tensors='pt', truncation=True,
                     max_length=max_length).to(device)
    inp_o2 = tok_obs(text, return_tensors='pt', truncation=True,
                     max_length=max_length).to(device)

    with torch.no_grad():
        logits_p = m_per(**inp_p).logits
        logits_o = m_obs(**inp_o2).logits

    # seq 길이 맞추기
    seq_len = min(logits_p.shape[1], logits_o.shape[1]) - 1
    if seq_len <= 0:
        return 0.0

    lp = logits_p[:, :seq_len, :]
    lo = logits_o[:, :seq_len, :]

    # vocab 크기 맞추기
    v  = min(lp.shape[-1], lo.shape[-1])
    lp = lp[..., :v]
    lo = lo[..., :v]

    p_per    = torch.softmax(lp, dim=-1)
    log_obs  = torch.log_softmax(lo, dim=-1)
    cross_loss = -(p_per * log_obs).sum(-1).mean().item()

    return loss_o / (cross_loss + 1e-10)


def run_experiment(df_input, m_obs, tok_obs, m_per, tok_per, device, desc=''):
    """전체 데이터프레임에 대해 Binoculars + PPL 점수 계산"""
    bino_scores = []
    ppl_scores  = []
    for text in tqdm(df_input['text'].tolist(), desc=desc):
        bino_scores.append(compute_binoculars(text, m_obs, tok_obs, m_per, tok_per, device))
        ppl_scores.append(compute_ppl(text, m_obs, tok_obs, device))
    return np.array(bino_scores), np.array(ppl_scores)

print('함수 정의 완료')

함수 정의 완료


## 실험 1: 영어 모델 쌍 (GPT2-medium + GPT2-large)
원본 Binoculars가 영어 기반이라는 걸 보이기 위한 베이스라인 실험

In [6]:
print('GPT2-medium (Observer) 로드 중...')
tok_en1 = AutoTokenizer.from_pretrained('gpt2-medium')
m_en1   = AutoModelForCausalLM.from_pretrained('gpt2-medium').eval().to(device)

print('GPT2-large (Performer) 로드 중...')
tok_en2 = AutoTokenizer.from_pretrained('gpt2-large')
m_en2   = AutoModelForCausalLM.from_pretrained('gpt2-large').eval().to(device)

print('완료!')

GPT2-medium (Observer) 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

완료!


In [7]:
# 샘플 100개로 영어 모델 실험 (시간 절약)
human_df = df[df['label']==0].sample(n=50, random_state=42)
llm_df   = df[df['label']==1].sample(n=50, random_state=42)
sample   = pd.concat([human_df, llm_df]).reset_index(drop=True)

bino_en, ppl_en = run_experiment(sample, m_en1, tok_en1, m_en2, tok_en2, device, desc='영어 모델 실험')
labels_sample   = np.array(sample['label'].tolist())

auc_en_bino = roc_auc_score(labels_sample, -bino_en)
auc_en_ppl  = roc_auc_score(labels_sample,  ppl_en)

print('\n===== 실험 1: 영어 모델 쌍 (n=100) =====')
print(f'PPL only      AUC: {auc_en_ppl:.4f}')
print(f'Binoculars    AUC: {auc_en_bino:.4f}')
print('>> 0.5에 가까울수록 한국어에서 작동 안 함')

영어 모델 실험: 100%|██████████| 100/100 [00:28<00:00,  3.54it/s]


===== 실험 1: 영어 모델 쌍 (n=100) =====
PPL only      AUC: 0.6444
Binoculars    AUC: 0.4176
>> 0.5에 가까울수록 한국어에서 작동 안 함


In [8]:
# 영어 모델 메모리 해제
del m_en1, m_en2
torch.cuda.empty_cache()
print('영어 모델 해제 완료')

영어 모델 해제 완료


## 실험 2 & 3: Ko-Binoculars (Polyglot-Ko 1.3B + 3.8B)
한국어 모델 쌍으로 교체한 Ko-Binoculars 전체 실험

In [9]:
print('Polyglot-Ko-1.3B (Observer) 로드 중...')
tok1 = AutoTokenizer.from_pretrained('EleutherAI/polyglot-ko-1.3b')
m1   = AutoModelForCausalLM.from_pretrained('EleutherAI/polyglot-ko-1.3b').eval().to(device)

print('Polyglot-Ko-3.8B (Performer) 로드 중...')
tok2 = AutoTokenizer.from_pretrained('EleutherAI/polyglot-ko-3.8b')
m2   = AutoModelForCausalLM.from_pretrained('EleutherAI/polyglot-ko-3.8b').eval().to(device)

print(f'완료! 디바이스: {device}')

Polyglot-Ko-1.3B (Observer) 로드 중...


config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

완료! 디바이스: cuda


In [10]:
# 전체 데이터 (2094개) 실험
bino_all, ppl_all = run_experiment(df, m1, tok1, m2, tok2, device, desc='Ko-Binoculars 전체 데이터')
labels_all = np.array(df['label'].tolist())

auc_bino = roc_auc_score(labels_all, -bino_all)
auc_ppl  = roc_auc_score(labels_all,  ppl_all)

print('\n===== 실험 2: Ko-Binoculars 전체 데이터 (n=2094) =====')
print(f'PPL only          AUC: {auc_ppl:.4f}')
print(f'Ko-Binoculars     AUC: {auc_bino:.4f}')
print(f'\nHuman 평균 점수: {bino_all[labels_all==0].mean():.4f}')
print(f'LLM   평균 점수: {bino_all[labels_all==1].mean():.4f}')

Ko-Binoculars 전체 데이터: 100%|██████████| 2094/2094 [07:29<00:00,  4.66it/s]


===== 실험 2: Ko-Binoculars 전체 데이터 (n=2094) =====
PPL only          AUC: 0.4176
Ko-Binoculars     AUC: 0.6460

Human 평균 점수: 1.0247
LLM   평균 점수: 0.9406


In [11]:
# 장르별 분석
print('\n===== 실험 3: 장르별 AUC (Ko-Binoculars, 전체 데이터) =====')

genres = {'essay': essay, 'abstract': abstract, 'poetry': poetry}
genre_results = {}

for genre_name, genre_data in genres.items():
    gdf   = pd.DataFrame(genre_data)
    texts = set(gdf['text'].tolist())
    idx   = [i for i, t in enumerate(df['text'].tolist()) if t in texts]

    g_scores = bino_all[idx]
    g_labels = labels_all[idx]

    if len(set(g_labels)) == 2:
        auc = roc_auc_score(g_labels, -g_scores)
        genre_results[genre_name] = auc
        print(f'  {genre_name:10s}  AUC: {auc:.4f}  '
              f'(human: {g_scores[g_labels==0].mean():.4f} / '
              f'llm: {g_scores[g_labels==1].mean():.4f})  n={len(g_labels)}')


===== 실험 3: 장르별 AUC (Ko-Binoculars, 전체 데이터) =====
  essay       AUC: 0.9680  (human: 0.9403 / llm: 0.7797)  n=771
  abstract    AUC: 0.7631  (human: 0.9436 / llm: 0.8873)  n=378
  poetry      AUC: 0.6879  (human: 1.1485 / llm: 1.0859)  n=945


## 실험 4: Ablation Study — 모델 쌍 크기 조합

In [12]:
# 샘플 100개 고정
human_df = df[df['label']==0].sample(n=50, random_state=42)
llm_df   = df[df['label']==1].sample(n=50, random_state=42)
sample   = pd.concat([human_df, llm_df]).reset_index(drop=True)
labels_s = np.array(sample['label'].tolist())

ablation_results = {}

# 조합 1: 1.3b + 1.3b (같은 모델)
print('[1/3] polyglot 1.3b + 1.3b (same)...')
scores = [compute_binoculars(t, m1, tok1, m1, tok1, device)
          for t in tqdm(sample['text'].tolist())]
ablation_results['1.3b + 1.3b (same)'] = roc_auc_score(labels_s, -np.array(scores))

# 조합 2: 1.3b + 3.8b (제안)
print('[2/3] polyglot 1.3b + 3.8b (제안)...')
scores = [compute_binoculars(t, m1, tok1, m2, tok2, device)
          for t in tqdm(sample['text'].tolist())]
ablation_results['1.3b + 3.8b (제안)'] = roc_auc_score(labels_s, -np.array(scores))

# 조합 3: 3.8b + 1.3b (역방향)
print('[3/3] polyglot 3.8b + 1.3b (역방향)...')
scores = [compute_binoculars(t, m2, tok2, m1, tok1, device)
          for t in tqdm(sample['text'].tolist())]
ablation_results['3.8b + 1.3b (역방향)'] = roc_auc_score(labels_s, -np.array(scores))

print('\n===== 실험 4: Ablation Study =====')
print(f'{"조합":<25} AUC')
print('-' * 35)
for k, v in ablation_results.items():
    marker = ' ← 최고' if v == max(ablation_results.values()) else ''
    print(f'{k:<25} {v:.4f}{marker}')

[1/3] polyglot 1.3b + 1.3b (same)...


100%|██████████| 100/100 [00:11<00:00,  8.69it/s]


[2/3] polyglot 1.3b + 3.8b (제안)...


100%|██████████| 100/100 [00:19<00:00,  5.21it/s]


[3/3] polyglot 3.8b + 1.3b (역방향)...


100%|██████████| 100/100 [00:26<00:00,  3.72it/s]


===== 실험 4: Ablation Study =====
조합                        AUC
-----------------------------------
1.3b + 1.3b (same)        0.6428
1.3b + 3.8b (제안)          0.6964 ← 최고
3.8b + 1.3b (역방향)         0.6508


## 최종 결과 정리

In [13]:
print('=' * 60)
print('Ko-Binoculars 전체 실험 결과 요약')
print('=' * 60)

print('\n[모델 쌍 비교]')
print(f'{"방법":<35} {"AUC":>8}')
print('-' * 45)
print(f'{"영어 Binoculars (GPT2-medium+large)":<35} {auc_en_bino:>8.4f}')
print(f'{"PPL only (polyglot-1.3b)":<35} {auc_ppl:>8.4f}')
for k, v in ablation_results.items():
    label = f'Ko-Binoculars ({k})'
    print(f'{label:<35} {v:>8.4f}')

print('\n[Ko-Binoculars 장르별 AUC (전체 데이터, 1.3b+3.8b)]')
print(f'{"장르":<15} {"AUC":>8}')
print('-' * 25)
for genre, auc in genre_results.items():
    print(f'{genre:<15} {auc:>8.4f}')
print(f'{"전체":<15} {auc_bino:>8.4f}')

print('\n[KatFishNet 논문 수치와 비교 (Park et al., ACL 2025)]')
comparison = [
    ('DetectGPT',              False, 52.78, '-',   67.04, 66.02),
    ('LLM Paraphrasing',       False, 92.08, 70.80, 71.32, 81.27),
    ('Fine-tuning (RoBERTa)',   True,  66.77, 50.70, 60.35, 65.93),
    ('KatFishNet (Punct.)',     True,  97.57, 78.99, 62.65, 94.88),
    ('Ko-Binoculars (ours)',    False,
     genre_results.get('essay',0)*100,
     genre_results.get('abstract',0)*100,
     genre_results.get('poetry',0)*100,
     auc_bino*100),
]
print(f'{"방법":<28} {"학습":>4} {"Essay":>7} {"Abst":>7} {"Poetry":>7} {"평균":>7}')
print('-' * 60)
for row in comparison:
    name, train, e, a, p, avg = row
    t = 'O' if train else 'X'
    e_s  = f'{e:.2f}'  if isinstance(e, float)  else e
    a_s  = f'{a:.2f}'  if isinstance(a, float)  else a
    p_s  = f'{p:.2f}'  if isinstance(p, float)  else p
    av_s = f'{avg:.2f}' if isinstance(avg, float) else avg
    print(f'{name:<28} {t:>4} {e_s:>7} {a_s:>7} {p_s:>7} {av_s:>7}')

print('\n* KatFishNet 수치는 Park et al. (ACL 2025) Table 3에서 인용')
print('* Ko-Binoculars는 본 실험 결과 (n=2094)')

Ko-Binoculars 전체 실험 결과 요약

[모델 쌍 비교]
방법                                       AUC
---------------------------------------------
영어 Binoculars (GPT2-medium+large)     0.4176
PPL only (polyglot-1.3b)              0.4176
Ko-Binoculars (1.3b + 1.3b (same))    0.6428
Ko-Binoculars (1.3b + 3.8b (제안))      0.6964
Ko-Binoculars (3.8b + 1.3b (역방향))     0.6508

[Ko-Binoculars 장르별 AUC (전체 데이터, 1.3b+3.8b)]
장르                   AUC
-------------------------
essay             0.9680
abstract          0.7631
poetry            0.6879
전체                0.6460

[KatFishNet 논문 수치와 비교 (Park et al., ACL 2025)]
방법                             학습   Essay    Abst  Poetry      평균
------------------------------------------------------------
DetectGPT                       X   52.78       -   67.04   66.02
LLM Paraphrasing                X   92.08   70.80   71.32   81.27
Fine-tuning (RoBERTa)           O   66.77   50.70   60.35   65.93
KatFishNet (Punct.)             O   97.57   78.99   62.65   94.88
Ko-Binocular